# RAG Pipeline - Exp3

* About
  * More modularized pipeline works for different data input
  * Deeper evaluation in retrieval and answer generation

In [10]:
%load_ext autoreload
%autoreload 2

import os
import yaml
os.environ.pop("OPENAI_API_KEY", None)

from datasets import load_dataset
from llama_index.core import Document
import pandas as pd
import json

from utils import *

import warnings
warnings.filterwarnings('ignore')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Load Data

* This applies to any dataset that has:
  * `contexts`: ground truth referenced documents, list format
  * `ground_truths`, list format
  * `question`

In [2]:
fiqa_eval = load_dataset("explodinggradients/fiqa", "ragas_eval")['baseline']

output_dir = "fiqa_raw_text"
os.makedirs(output_dir, exist_ok=True)

rag_lst = []
documents = []  # to store documents for llamindex
for idx, record in enumerate(fiqa_eval):
    context = ''.join(record['contexts'])
    gt = ''.join(record['ground_truths'])
    if 'answer' in record.keys():
        ai0_answer = record['answer'].strip()
    else:
        ai0_answer = None

    with open(os.path.join(output_dir, f"{idx}.txt"), "w", encoding="utf-8") as f:
        f.write(context)
    rag_lst.append({
        'question': record['question'],
        'context': context,
        'context_ct': len(record['contexts']),
        'ground_truth': gt,
        'ai0_answer': ai0_answer
    })
    doc = Document(
        text=context,
        metadata={
            "doc_name": idx
        }
    )
    documents.append(doc)

rag_df = pd.DataFrame(rag_lst)
print(rag_df.shape, max(rag_df['context_ct']), min(rag_df['context_ct']))
print(len(documents))
rag_df.head()

(30, 5) 1 1
30


,question,context,context_ct,ground_truth,ai0_answer
0,How to deposit a cheque issued to an associate...,Just have the associate sign the back and then...,1,Have the check reissued to the proper payee.Ju...,The best way to deposit a cheque issued to an ...
1,Can I send a money order from USPS as a business?,Sure you can. You can fill in whatever you wa...,1,Sure you can. You can fill in whatever you wa...,"Yes, you can send a money order from USPS as a..."
2,1 EIN doing business under multiple business n...,You're confusing a lot of things here. Company...,1,You're confusing a lot of things here. Company...,"Yes, it is possible to have one EIN doing busi..."
3,Applying for and receiving business credit,Set up a meeting with the bank that handles yo...,1,"""I'm afraid the great myth of limited liabilit...",Applying for and receiving business credit can...
4,401k Transfer After Business Closure,The time horizon for your 401K/IRA is essentia...,1,You should probably consult an attorney. Howev...,If your employer has closed and you need to tr...


### RAG Pipeline

In [3]:
with open("all_rag_pipeline_config.yaml", "r", encoding="utf-8") as f:
     all_config= yaml.safe_load(f)

In [6]:
for i in range(len(all_config['embedding_models'])):
    embed_model_str = all_config['embedding_models'][i]
    indexing_storage_dir = all_config['indexing_storage_dirs'][i]
    print(f"Embedding model: {embed_model_str}")
    create_vector_index(documents, embed_model_str, indexing_storage_dir)

Embedding model: BAAI/bge-small-en-v1.5


Parsing nodes:   0%|          | 0/30 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/35 [00:00<?, ?it/s]

Index saved to ./storage_m1


In [8]:
retriever_params = all_config['retriever_params']
cfgs = []
for i in range(len(all_config['yaml_config_files'])):
    yaml_config_file = 'output/saved_configs/' + all_config['yaml_config_files'][i]
    cfgs.append(yaml_config_file)
    config = {
        'llm_model': all_config['llm_models'][i],
        'embedding_model': all_config['embedding_models'][i],
        'indexing_storage_dir': all_config['indexing_storage_dirs'][i],
        'output_file': all_config['output_files'][i],
        'retriever_params': retriever_params
    }

    with open(yaml_config_file, "w", encoding="utf-8") as f:
        yaml.safe_dump(config, f, sort_keys=False)

In [11]:
max_worker = os.cpu_count() - 1

await run_all_in_processes(cfgs, rag_lst, documents, max_workers=max_worker)